# Normalisation — verification

Checks the model's inputs and target before training. **What:** confirm the saved masks and
normalisation behave as intended. **Approach:** RGB ÷ 255 → [0,1]; destriped EI → [0,1] with the
train-skin p1/p99 stats (`src.normalization`); masks are binary 0/1 selectors. **Why:** the model
learns RGB → normalised EI on skin only, so inputs must be scaled and the target must fill [0,1] on
skin without excessive clipping. Images shown only for the display-permitted subjects (p012/p019/p027).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.io_utils import load_rgb
from src.normalization import normalize_rgb, normalize_ei, load_stats

PROCESSED     = (Path(config.__file__).resolve().parent / config.LOCAL_PROCESSED_DIR).resolve()
MASKS_DIR     = PROCESSED / "masks"
DESTRIPED_DIR = PROCESSED / "ei_maps_destriped"

manifest = pd.read_csv(PROCESSED / "manifest.csv")
stats    = load_stats(str(PROCESSED / "norm_stats.json"))
path_of  = {f"{r.subject_id}_{r.pose}_{r.view}": r.rgb_path for r in manifest.itertuples(index=False)}
DISCLOSURE = ["p012", "p019", "p027"]          # only subjects permitted for display

print("EI norm stats (train skin):", stats)
print("splits:", manifest.split.value_counts().to_dict())

## 1. Masks — overlay check

Binary 0/1 masks (saved by `scripts/compute_masks.py`) on the disclosure subjects, all profiles:
RGB · mask · RGB × mask. Confirms the mask is 1 on skin, 0 elsewhere, and aligns with the face.

In [ ]:
demo = [f"{s}_neutral_{v}" for s in DISCLOSURE for v in ["front", "left", "right"]]
fig, axes = plt.subplots(len(demo), 3, figsize=(11, 3.6*len(demo)))
for row, stem in enumerate(demo):
    rgb  = load_rgb(str(path_of[stem]))
    mask = np.load(MASKS_DIR / f"{stem}.npy")
    for ax, (im, t, cm) in zip(axes[row], [
        (rgb,  f"{stem} RGB", None),
        (mask, f"mask 0/1 ({mask.mean()*100:.0f}%)", "gray"),
        (rgb/255.0 * mask[:, :, None], "RGB x mask", None)]):
        ax.imshow(im, cmap=cm); ax.set_title(t, fontsize=9); ax.axis("off")
plt.suptitle("Masks — binary 0/1 selector, aligned to the face", y=1.002)
plt.tight_layout(); plt.show()

## 2. Normalised value ranges

RGB ÷ 255 must land in [0,1]; normalised EI must fill [0,1] on skin with little clipping. Per-image
skin ranges and clip fractions are aggregated over the whole dataset (numbers only, no images);
the histogram is the train-skin distribution that defines the scale.

**How to read the outputs:**
- **Table** = per-image values on skin pixels, averaged within each split.
  - `skin_min`/`skin_max` = 0.0/1.0 everywhere → every image touches both ends of the scale.
    Expected by construction (p1/p99 clip ~1% per side); a `skin_max` well below 1 would mean the
    scale is too wide (signal squeezed).
  - `clip_lo_pct`/`clip_hi_pct` = % of skin pixels clipped at 0/1 — the information actually lost.
    Train ≈ 1%/1% is a consistency check (true by definition of the percentiles). **Test and valid
    are the real result**: they were not used for the stats, so ~0.9–1.4% there means the
    train-derived scale transfers to unseen subjects. Concern threshold: several % (regions of
    real signal saturating).
- **RGB line**: values inside [0,1], float32 — min 0.024 just means no pure-black pixel in that image.
- **Histogram**: train-skin EI should spread across [0,1] with the bulk away from the 0/1 edges —
  confirms the target has usable dynamic range, not a spike or an edge pile-up.

**Verdict: pass** — clipping ~1% per side on all splits, scale fits unseen subjects.

In [ ]:
rng = np.random.default_rng(0)
rows, hist = [], []
for r in manifest.itertuples(index=False):
    stem = f"{r.subject_id}_{r.pose}_{r.view}"
    ei   = np.load(DESTRIPED_DIR / f"{stem}.npy")
    mask = np.load(MASKS_DIR / f"{stem}.npy").astype(bool)
    n    = normalize_ei(ei, stats)[mask]                    # normalised EI on skin pixels
    rows.append(dict(split=r.split, skin_min=float(n.min()), skin_max=float(n.max()),
                     clip_lo_pct=float((n <= 0).mean()*100), clip_hi_pct=float((n >= 1).mean()*100)))
    if r.split == "train":
        hist.append(n if n.size <= 20000 else rng.choice(n, 20000, replace=False))
diag = pd.DataFrame(rows)

print("normalised skin-EI range + clip% (mean per image, by split):")
print(diag.groupby("split")[["skin_min", "skin_max", "clip_lo_pct", "clip_hi_pct"]].mean().round(3).to_string())

# RGB range spot-check
rgb0 = normalize_rgb(load_rgb(str(path_of[f"{DISCLOSURE[0]}_neutral_front"])))
print(f"\nnormalised RGB range: [{rgb0.min():.3f}, {rgb0.max():.3f}]  dtype={rgb0.dtype}")

allv = np.concatenate(hist)
plt.figure(figsize=(7, 3.5))
plt.hist(allv, bins=60, color="steelblue")
plt.title("Normalised skin EI (train) — should fill [0,1], mass off the 0/1 edges")
plt.xlabel("normalised EI"); plt.ylabel("skin-pixel count"); plt.tight_layout(); plt.show()

## 3. The model's input and target, assembled

For the disclosure subjects: normalised RGB (input) and normalised EI × mask (target, [0,1]). Same
mask on both, so input and target are pixel-aligned — this is exactly what the loader will produce.

In [ ]:
fig, axes = plt.subplots(len(demo), 3, figsize=(12, 3.8*len(demo)))
for row, stem in enumerate(demo):
    rgb  = normalize_rgb(load_rgb(str(path_of[stem])))
    ei   = normalize_ei(np.load(DESTRIPED_DIR / f"{stem}.npy"), stats)
    mask = np.load(MASKS_DIR / f"{stem}.npy").astype(bool)
    target = np.where(mask, ei, np.nan)
    for ax, (im, t, cm, kw) in zip(axes[row], [
        (rgb,   f"{stem}\ninput: RGB/255", None, {}),
        (mask,  "mask 0/1", "gray", {}),
        (target, "target: norm EI x mask", "magma", dict(vmin=0, vmax=1))]):
        h = ax.imshow(im, cmap=cm, **kw); ax.set_title(t, fontsize=9); ax.axis("off")
        if cm == "magma":
            plt.colorbar(h, ax=ax, fraction=0.046)
plt.suptitle("Input (normalised RGB) and target (normalised EI x mask), pixel-aligned", y=1.003)
plt.tight_layout(); plt.show()

## Conclusion

- **Masks:** binary 0/1, aligned to the face, ~25% coverage — a clean region selector.
- **RGB:** ÷ 255 → [0,1].
- **EI target:** scaled to [0,1] with train-skin p1/p99; the train-skin histogram fills the range and
  clipping stays small, so the scale is well-matched to the erythema signal.
- **Contract confirmed:** input = normalised RGB (full frame), target = normalised destriped EI, loss
  on `mask==1` only. Ready for Stage 4 (U-Net: RGB → normalised EI, masked loss).